# Day 4.2 — Single-Reviewer Baseline

## Before you begin

### Learning outcomes

- Call a reviewer through one provider contract that both a real model and an offline mock satisfy.
- Read the telemetry a run actually produced instead of an estimate.
- Measure the baseline that every later architecture must beat.

Architecture reference: [Day 4 diagrams D12](../diagrams/source/day_04.md)

### Expected observation

One reviewer finds 5 of the 9 seeded defects in mock mode, and the run reports the tokens the provider says it used.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/review_team"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")


## Step 1 — Load the artifact


In [ ]:
# The artifact under review and the instructor's answer key.
# Both are plain files; nothing here is secret from you, but the answer key is never
# put into a reviewer's prompt.
ARTIFACT_PATH = PROJECT_ROOT / "data" / "seeded_artifact" / "order_service.py"
GOLDEN_PATH   = PROJECT_ROOT / "data" / "golden_defects.json"

SOURCE = ARTIFACT_PATH.read_text(encoding="utf-8")   # the shared, immutable artifact

print("Artifact file :", ARTIFACT_PATH.name)
print("Artifact lines:", len(SOURCE.splitlines()))
print("Answer key    :", GOLDEN_PATH.name)


## Step 2 — One contract, two implementations

Every reviewer today — real or mock — is called the same way:

```python
findings, usage = provider.review(source, role)
```

`role` is `"general"` for this baseline. Because the contract is identical, we can swap the reviewer without touching the architecture we are testing.


In [ ]:
# Build the reviewer. This is the ONLY place the notebook decides live vs mock.
from review_team import FallbackReviewer, MockStructuredReviewer, OpenRouterReviewer

SCENARIO = "blind_spots"          # which blind spots the scripted reviewer has

if LIVE:
    # FallbackReviewer tries the real model and, if the call fails for any reason,
    # prints one line and uses the mock for that call so the lesson never stops.
    provider = FallbackReviewer(OpenRouterReviewer(), MockStructuredReviewer(SCENARIO))
else:
    provider = MockStructuredReviewer(SCENARIO)

print("Reviewer provider:", type(provider).__name__)
print("Scenario         :", SCENARIO)


## Step 3 — Run the single reviewer

`run_single_reviewer` makes exactly one call, applies no tools and runs no supervisor. It is the simplest system that could possibly work.


In [ ]:
from review_team import run_single_reviewer

single = run_single_reviewer(SOURCE, provider)

print("System     :", single.system)
print("Model calls:", single.model_calls)
print("Findings   :", len(single.findings), "\n")

for finding in single.findings:
    print(f"line {finding.line:>3} | {finding.category:<15} | {finding.severity:<8} | "
          f"{finding.title}")


## Step 4 — Read the telemetry, do not guess it

The token numbers below come from the provider's own usage report, which the run stored in its trace. A step that never calls a model reports zero — you will see one of those in the next notebook.


In [ ]:
step = single.trace[0]                  # the run's only step
usage = step["usage"]                  # what the provider said it consumed

print("Step             :", step["step"])
print("Model            :", usage["model"])
print("Live call?       :", usage["live"])
print("Prompt tokens    :", usage["prompt_tokens"])
print("Completion tokens:", usage["completion_tokens"])
print("Total tokens     :", single.total_tokens)
print("Cost reported    : $%.6f" % single.cost_usd)
print("Elapsed ms       : %.2f" % single.elapsed_ms)


## Step 5 — Score the baseline

Now the golden set earns its keep. `missed` is the list this whole day exists to shrink — or to prove we cannot shrink it economically.


In [ ]:
from review_team import evaluate

row = evaluate(single, GOLDEN_PATH)

print("Found      : %d / %d" % (row["found"], row["known_defects"]))
print("Recall     :", row["recall"])
print("False positives:", row["false_positives"])
print("Duplicates :", row["duplicates"])
print("Missed     :", row["missed"])


## Step 6 — Be honest about what this reviewer is

In mock mode the reviewer is **scripted**, not intelligent. Its blind spots are a parameter we chose, so the classroom result is reproducible on any laptop with no API key. Nothing here is a claim about how good real language models are at code review — it is a controlled way to compare *architectures* while holding the reviewer constant.


In [ ]:
from review_team import SCENARIOS

for scenario, roles in SCENARIOS.items():
    print(scenario, "-> general reviewer can see", len(roles["general"]), "of 9 defects")
print("\nCurrently using:", SCENARIO)


### Try it yourself

Predict: if we swap in the `strong_generalist` reviewer — same architecture, one call, no specialists — how many of the 9 defects will it find? Write your number down, then run the worked solution.


In [ ]:
# --- Worked solution ---
# Same architecture (run_single_reviewer), different reviewer. Only the blind spots move.
strong_provider = MockStructuredReviewer("strong_generalist")
strong = run_single_reviewer(SOURCE, strong_provider)
strong_row = evaluate(strong, GOLDEN_PATH)

print("blind_spots       reviewer found: %d / 9  (missed %s)"
      % (row["found"], row["missed"]))
print("strong_generalist reviewer found: %d / 9  (missed %s)"
      % (strong_row["found"], strong_row["missed"]))
print()
print("Both runs used", strong.model_calls, "model call. The architecture did not change;")
print("the reviewer did. Remember this when we start adding agents.")


### Checkpoint

**1. Why does every reviewer — mock, live, and the fake one in the tests — go through the same `provider.review(source, role)` contract?**

<details><summary>Show answer</summary>

Because it lets us change one variable at a time. If the mock and the live model plug into the same slot, a difference in results comes from the architecture or the reviewer, never from rewriting the pipeline. It is also what makes an offline, zero-cost classroom path possible.

</details>

**2. The run reports `prompt_tokens` from the provider rather than estimating them from the length of the source. Why does that distinction matter?**

<details><summary>Show answer</summary>

An estimate printed in a results table looks exactly like a measurement. If a step that made no call still reports a plausible token count, every cost comparison built on that table is fiction. Measured telemetry — or an explicit zero labelled "no model call" — is the only honest option.

</details>

### Recap

- Limitation we saw: one general reviewer covering everything found 5 of 9 seeded defects and missed a whole category's worth.
- Layer we added: a single provider contract with real usage telemetry, plus a scored baseline run.
- Evidence it worked: 5/9 recall, 1 model call, and a printed `missed` list that later systems have to shrink.
